# 03 - Model Training, Evaluation & Explainability
Compares several models using a metric set that prioritizes Recall/PR-AUC (see `src/evaluation.py`, `src/models.py`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fossbk-spec/heart-failure/blob/main/notebooks/03_model_training_and_evaluation.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/fossbk-spec/heart-failure/blob/main/notebooks/03_model_training_and_evaluation.ipynb)

In [1]:
# Runs offline on Google Colab / Kaggle Notebooks: no internet/Kaggle token needed.
# The dataset CSV is already bundled in the repo at data/raw/.
import os, sys, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    if not os.path.exists("heart-failure"):
        subprocess.run(["git", "clone", "--depth", "1",
                         "https://github.com/fossbk-spec/heart-failure.git"], check=True)
    os.chdir("heart-failure")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # run from the project root locally
sys.path.insert(0, os.getcwd())


In [2]:
from src.data_loader import load_raw_data
from src.train import run_experiment

df = load_raw_data()
table, cv_tables = run_experiment(
    df,
    model_names=['logistic_regression', 'random_forest', 'gradient_boosting', 'stacking'],
    use_smote=True,
    min_recall=0.8,  # clinical operating point: keep >=80% of deaths flagged
    output_dir=None,
)

[OK] ZERO DATA LEAKAGE confirmed (Train=191 IDs | Validation=48 IDs | Test=60 IDs).

--- Data distribution after patient-level split ---
split  n_samples  n_patients  n_positive  positive_rate
train        191         191          62       0.324607
  val         48          48          15       0.312500
 test         60          60          19       0.316667

=== logistic_regression (smote=True, class_weight=None) ===
logistic_regression @ threshold=0.377
  TP=10  FN=9  FP=13  TN=28
  recall          = 0.5263
  pr_auc          = 0.5616
  f2              = 0.5051
  f1              = 0.4762
  precision       = 0.4348
  roc_auc         = 0.6919
  specificity     = 0.6829
  balanced_accuracy= 0.6046
  accuracy        = 0.6333
  null_accuracy   = 0.6833
  f2_flag_all     = 0.6985
  brier           = 0.2290
  n_false_negative= 9.0000
  n_true_positive = 10.0000
  n_false_positive= 13.0000
  CV (train+val):
    {'recall': '0.8558 ± 0.0746', 'pr_auc': '0.6284 ± 0.0344', 'f2': '0.7223 ± 0.0309'

## Comparison table (sorted by Recall)
`f2_flag_all` is the F2 of flagging every patient — at a 32% death rate it is already ~0.70, so F2 alone cannot distinguish a useful model from a trivial one. Read Recall together with Specificity and PR-AUC.

In [3]:
cols = ['model','threshold','recall','specificity','pr_auc','f2','f2_flag_all','precision','roc_auc','n_false_negative']
table[cols].round(4)

,model,threshold,recall,specificity,pr_auc,f2,f2_flag_all,precision,roc_auc,n_false_negative
0,stacking,0.2613,0.7368,0.6585,0.6306,0.6731,0.6985,0.5000,0.7587,5.0
1,gradient_boosting,0.2009,0.7368,0.6585,0.4629,0.6731,0.6985,0.5000,0.6906,5.0
2,random_forest,0.3350,0.6842,0.6341,0.5438,0.6250,0.6985,0.4643,0.7433,6.0
3,logistic_regression,0.3766,0.5263,0.6829,0.5616,0.5051,0.6985,0.4348,0.6919,9.0


## Grouped cross-validation (more reliable than the 60-patient test set)

In [4]:
from src.evaluation import summarize_cv
import pandas as pd
pd.DataFrame({name: summarize_cv(cv)[['recall','specificity','pr_auc','roc_auc']]
              for name, cv in cv_tables.items()}).T

,recall,specificity,pr_auc,roc_auc
logistic_regression,0.8558 ± 0.0746,0.4926 ± 0.1487,0.6284 ± 0.0344,0.7682 ± 0.0432
random_forest,0.7883 ± 0.1457,0.6847 ± 0.1391,0.6694 ± 0.0398,0.7780 ± 0.0546
gradient_boosting,0.8283 ± 0.1307,0.4559 ± 0.1432,0.6296 ± 0.1068,0.7439 ± 0.0594
stacking,0.7767 ± 0.1037,0.6172 ± 0.1446,0.6323 ± 0.1062,0.7621 ± 0.0776


## Clinical explainability (XAI): SHAP + Odds Ratio

In [5]:
from src.data_loader import split_features_target
from src.preprocessing import patient_level_split, make_full_pipeline
from src.models import get_model
from src import explainability

split = patient_level_split(df)
X_train, y_train = split_features_target(split.train)

rf_pipeline = make_full_pipeline(get_model('random_forest', class_weight='balanced'), scale=False)
rf_pipeline.fit(X_train, y_train)
values, importance = explainability.shap_summary(
    rf_pipeline, X_train, save_path='reports/figures/shap_summary_rf.png'
)
importance

[OK] ZERO DATA LEAKAGE confirmed (Train=191 IDs | Validation=48 IDs | Test=60 IDs).


D:\Antigravity\heart-failure\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,feature,mean_abs_shap
0,serum_creatinine,0.131109
1,ejection_fraction,0.088862
2,serum_sodium,0.054315
3,high_blood_pressure,0.037928
4,age,0.035407
5,creatinine_phosphokinase,0.026240
6,smoking,0.026236
7,diabetes,0.024991
8,anaemia,0.021475
9,platelets,0.019428


In [6]:
lr_pipeline = make_full_pipeline(get_model('logistic_regression'))
lr_pipeline.fit(X_train, y_train)
explainability.logistic_odds_ratios(lr_pipeline, X_train)

,feature,coefficient,odds_ratio,direction,or_ci_lower,or_ci_upper,significant
0,serum_creatinine,1.300818,3.672301,increases risk,2.056204,6.558589,True
1,high_blood_pressure,0.935489,2.548460,increases risk,1.261161,5.149741,True
2,smoking,0.933584,2.543609,increases risk,1.046543,6.182208,True
3,ejection_fraction,-0.844259,0.429876,decreases risk,0.275195,0.671501,True
4,diabetes,0.822941,2.277187,increases risk,1.163169,4.458147,True
5,anaemia,0.762834,2.144345,increases risk,1.101254,4.175434,True
6,creatinine_phosphokinase,0.388088,1.474160,increases risk,1.011946,2.147494,True
7,age,0.338473,1.402804,increases risk,0.946431,2.079242,False
8,serum_sodium,-0.328582,0.719944,decreases risk,0.482661,1.073879,False
9,sex,-0.270537,0.762970,decreases risk,0.366670,1.587592,False
